# ch03 练习：数据探索

> 共 8 道基础题 + 1 道综合题（拆 3 小步，合计 12 个空）
> 建议限时：**25 分钟**

数据：`data/device_defects.csv`（270 行 x 13 列，含缺失、重复、异常值）

**这章练的是「不漏步」**。每道题都对应竞赛评分表里一个 2 分小项，
写不出来就是那 2 分没了。

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

dev = pd.read_csv("data/device_defects.csv")
print("读入 ->", dev.shape)

---

### TODO(1) 规模：行数与列数

分别取出行数、列数存入 `n_rows`、`n_cols`。
（不要写死 270 / 13，要用属性取。）

In [ ]:
n_rows = dev.shape[0]
n_cols = dev.shape[1]

# ---- 验收 ----
assert n_rows == 270, f"行数应为 270，实际 {n_rows}"
assert n_cols == 13, f"列数应为 13，实际 {n_cols}"
assert n_cols == len(dev.columns), "列数应该等于列名个数"
print("TODO(1) 通过 ->", n_rows, "行 x", n_cols, "列")

### TODO(2) 类型分布 + 按类型分组取列

1. 统计每种 dtype 各有几列，存入 `dtype_counts`（按数量降序的 Series）
2. 取数值列名列表存入 `num_cols`，文本列名列表存入 `txt_cols`

**注意**：pandas 3.0 里字符串列的 dtype 是 `str` 而不是 `object`。
想「取到所有非数值列」，用 `exclude="number"` 最稳。

In [ ]:
dtype_counts = dev.dtypes.astype(str).value_counts()
num_cols = list(dev.select_dtypes("number").columns)
txt_cols = list(dev.select_dtypes(exclude="number").columns)

# ---- 验收 ----
assert dtype_counts.to_dict() == {"str": 9, "float64": 3, "int64": 1}, (
    f"类型分布不对：{dtype_counts.to_dict()}"
)
assert len(num_cols) == 4, f"数值列应为 4 列，实际 {len(num_cols)}：{num_cols}"
assert len(txt_cols) == 9, f"文本列应为 9 列，实际 {len(txt_cols)}：{txt_cols}"
assert set(num_cols) == {"记录ID", "负荷值", "温度", "湿度"}, "数值列清单不对"
print("TODO(2) 通过 -> dtype 分布:", dtype_counts.to_dict())

### TODO(3) `describe`：数值列体检

1. 对数值列做 `describe()`，结果存入 `desc_num`
2. 从里面取出 `负荷值` 列的**中位数**（`50%` 那一行），存入 `load_median`

In [ ]:
desc_num = dev.describe()
load_median = desc_num.loc["50%", "负荷值"]

# ---- 验收 ----
assert desc_num.shape == (8, 4), f"describe() 形状应为 (8, 4)，实际 {desc_num.shape}"
assert list(desc_num.index) == ["count", "mean", "std", "min", "25%", "50%", "75%", "max"], "统计量行名不对"
assert abs(load_median - 87.34) < 1e-6, f"负荷值中位数应为 87.34，实际 {load_median}"
print("TODO(3) 通过 -> 负荷值中位数 =", load_median)

### TODO(4) `describe(include="all")`：把类别列也统计进来

对全部列做 `describe(include="all")`，存入 `desc_all`，
然后取出 **`缺陷类型`** 这一列的 `unique` 和 `top`，分别存入 `defect_unique`、`defect_top`。

In [ ]:
desc_all = dev.describe(include="all")
defect_unique = desc_all.loc["unique", "缺陷类型"]
defect_top = desc_all.loc["top", "缺陷类型"]

# ---- 验收 ----
assert desc_all.shape == (11, 13), f"形状应为 (11, 13)，实际 {desc_all.shape}"
assert defect_unique == 10, f"缺陷类型应有 10 种写法，实际 {defect_unique}"
assert defect_top == "发热", f"出现最多的缺陷类型应为「发热」，实际 {defect_top}"
print("TODO(4) 通过 -> 缺陷类型 %d 种，最多的是 %r" % (defect_unique, defect_top))

### TODO(5) 缺失普查

1. 逐列缺失计数存入 `miss`（Series）
2. 只保留缺失数 > 0 的列，按缺失数**降序**存入 `miss_sorted`
3. 全表缺失单元格总数存入 `n_missing_cells`

**坑位**：`isna().sum()` 给的是逐列结果，别把链式调用写反。

In [ ]:
miss = dev.isna().sum()
miss_sorted = miss[miss > 0].sort_values(ascending=False)
n_missing_cells = int(miss.sum())

# ---- 验收 ----
assert len(miss) == 13, "miss 应该是「逐列」的，长度等于列数"
assert miss_sorted.to_dict() == {"备注": 196, "温度": 23, "负荷值": 21}, (
    f"有缺失的列不对：{miss_sorted.to_dict()}"
)
assert n_missing_cells == 240, f"全表缺失单元格应为 240，实际 {n_missing_cells}"
print("TODO(5) 通过 -> 缺失列:", miss_sorted.to_dict())

### TODO(6) 重复行普查

1. 完全重复行数存入 `n_dup`
2. 按业务键（`台区编号`+`线路名称`+`设备类型`+`设备编号`+`缺陷类型`+`缺陷等级`+`发现日期`）
   去重后的重复行数存入 `n_dup_biz`

**注意**：业务键重复里既有「完全重复行」，也有「内容几乎相同但 `记录ID`/`备注` 不同」的行。

In [ ]:
biz_key = ["台区编号", "线路名称", "设备类型", "设备编号", "缺陷类型", "缺陷等级", "发现日期"]

n_dup = int(dev.duplicated().sum())
n_dup_biz = int(dev.duplicated(subset=biz_key).sum())

# ---- 验收 ----
assert n_dup == 6, f"完全重复行应为 6，实际 {n_dup}"
assert n_dup_biz == 10, f"业务键重复行应为 10，实际 {n_dup_biz}"
assert n_dup_biz > n_dup, "业务重复应该比完全重复多（多出的是记录ID不同的那些）"
print(f"TODO(6) 通过 -> 完全重复 {n_dup} 行，业务重复 {n_dup_biz} 行")

### TODO(7) `value_counts(dropna=False)`：类别分布

对 `处理状态` 列做 `value_counts(dropna=False)`，存入 `status_vc`。

这题的重点是**看清同一业务含义有几种写法**：
`已处理` / `已 处理` / `处理完成` / `done` 其实是同一类。

In [ ]:
status_vc = dev["处理状态"].value_counts(dropna=False)

# ---- 验收 ----
assert len(status_vc) == 8, f"处理状态应出现 8 种写法，实际 {len(status_vc)}"
assert status_vc.index[0] == "已处理", "出现最多的应该还是标准写法「已处理」"
assert set(status_vc.index) >= {"已处理", "已 处理", "处理完成", "done", "未处理", "待处理"}, "少了几种写法"
print("TODO(7) 通过 -> 该列 NunIque 应为 8：")
print(status_vc.to_string())

### TODO(8) 均值 vs 中位数：嗅出异常值

对 `负荷值` 列分别算 `mean` 和 `median`，存入 `load_mean`、`load_med_raw`。

然后**剔除 >= 1000 的极端值**后再算一次中位数，存入 `load_med_clean`。

（提示：`mean` 会被 `9999` 哨兵值严重拉高，`median` 几乎不受影响。）

In [ ]:
load_mean = dev["负荷值"].mean()
load_med_raw = dev["负荷值"].median()
load_med_clean = dev.loc[dev["负荷值"] < 1000, "负荷值"].median()

# ---- 验收 ----
assert load_mean > 300, f"均值应被哨兵值拉高到 300 以上，实际 {load_mean:.2f}"
assert load_med_raw < 100, f"中位数应远小于均值，实际 {load_med_raw:.2f}"
assert abs(load_med_raw - load_med_clean) < 5, (
    f"中位数几乎不受极端值影响，两者应很接近：{load_med_raw:.2f} vs {load_med_clean:.2f}"
)
assert load_mean / load_med_raw > 3, "均值/中位数应该差 3 倍以上"
print(f"TODO(8) 通过 -> mean={load_mean:.2f}  median={load_med_raw:.2f}  "
      f"clean_median={load_med_clean:.2f}")

---

## 综合题：把探索动作打包进一份「体检报告」

目标：写出一段能在**任何** DataFrame 上复用、并且**逐项对应竞赛计分点**的探索代码。

| 步骤 | 产物 | 对应评分点 |
|---|---|---|
| ① 缺失总览 | `miss_total` | 数据探索 · 缺失量 |
| ② 主导类别 | `top_defect` | 数据探索 · 类别分布 |
| ③ 组装报告 | `report` | 数据探索 · 结论输出 |

#### 综合 ① 缺失总览

In [ ]:
miss_total = int(dev.isna().sum().sum())

# ---- 验收 ----
assert miss_total == 240, f"缺失单元格总数应为 240，实际 {miss_total}"
print("综合 ① 通过 -> 全表缺失 =", miss_total)

#### 综合 ② 主导类别

In [ ]:
top_defect = dev["缺陷类型"].value_counts().index[0]

# ---- 验收 ----
assert top_defect == "发热", f"出现最多的缺陷类型应为「发热」，实际 {top_defect}"
assert isinstance(top_defect, str), "应该是字符串取值"
print("综合 ② 通过 -> 主导缺陷类型 =", top_defect)

#### 综合 ③ 组装报告字典

把上面算出来的各项指标组装成一个 `report` 字典，键必须是：

```
n_rows / n_cols / n_num_cols / n_text_cols / n_missing_cells / n_dup_rows / top_defect
```

In [ ]:
report = {
    "n_rows": n_rows,
    "n_cols": n_cols,
    "n_num_cols": len(num_cols),
    "n_text_cols": len(txt_cols),
    "n_missing_cells": miss_total,
    "n_dup_rows": n_dup,
    "top_defect": top_defect,
}

# ---- 验收 ----
expect_keys = {
    "n_rows", "n_cols", "n_num_cols", "n_text_cols",
    "n_missing_cells", "n_dup_rows", "top_defect",
}
assert isinstance(report, dict), "report 应该是字典"
assert set(report) == expect_keys, f"键不对，多的是 {set(report) - expect_keys}，缺的是 {expect_keys - set(report)}"
assert report["n_rows"] == 270 and report["n_cols"] == 13
assert report["n_num_cols"] == 4 and report["n_text_cols"] == 9
assert report["n_num_cols"] + report["n_text_cols"] == report["n_cols"], "两类列数加起来应等于总列数"
assert report["n_missing_cells"] == 240, "缺失总量不对"
assert report["n_dup_rows"] == 6, "重复行数不对"
assert report["top_defect"] == "发热", "主导类别不对"
print("综合 ③ 通过 ->")
for k, v in report.items():
    print(f"  {k:18s} = {v!r}")

---

### 复盘提问

**复盘问自己**：如果现在给你一份全新的数据，你能不能在 3 分钟内
不看笔记把上面 8 个探索动作默写出来？默不出来就说明还没形成肌肉记忆——
回 `_notes/错题本.md` 记一笔，明天再默一次。